# Double Machine Learning — ATE and subgroup heterogeneity

For each of the four outcomes (pressures, tackles, fouls, and their sum):
the partial-linear ATE (cross-fitted, orthogonal score, match-clustered SE),
followed by single-moderator subgroup ATEs on the pre-specified moderators
defined in `analysis_config.py` ($Z$). The confounder set $W$ and the moderator
set $Z$ share the categorical dummies (position, venue, gender, competition
format); $W$ also contains pre-window event counts, while $Z$ contains the
half-time score differential.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
import statsmodels.formula.api as smf
from scipy.stats import norm

df = pd.read_csv("data/analysis_frame.csv", low_memory=False)
DVS = ["post_n_pressure", "post_n_tackle", "post_n_foul_committed", "post_n_def_events"]
TREATMENT = "treat_yellow_card"
t = df[TREATMENT].astype(int).values
groups = df["match_id"].values
print(f"Sample: {len(df):,} player-matches | treated {t.sum():,} ({t.mean():.2%})")

Sample: 80,226 player-matches | treated 3,210 (4.00%)


In [2]:
import sys; sys.path.insert(0, "src")
# W (confounders) and Z (moderators) are defined in analysis_config.py — one
# place to edit; the notebook and the figure script both rebuild from it.
from analysis_config import build_W_Z, Z_NUM, CATS
W, Z, W_NUM, _, _, catdum = build_W_Z(df)
Wv, Zv = W.values, Z.values
print(f"W: {W.shape[1]} cols | Z: {Z.shape[1]} cols  "
      f"({len(Z_NUM)} num + {catdum.shape[1]} dummies)")
assert not any(c.startswith("ht_") for c in W.columns), "W must not contain ht_*"
assert not any(c.startswith("str_") for c in Z.columns), "Z must not contain str_*"
assert not any(c.startswith("ht_player_") for c in Z.columns), "Z must not contain ht_player_*"

W: 61 cols | Z: 9 cols  (4 num + 5 dummies)


In [3]:
MODEL_Y = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05,
                                        min_samples_leaf=200, random_state=0)
MODEL_T = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                         min_samples_leaf=200, random_state=0)
cf = GroupKFold(5)
# propensity cross-fit ONCE (outcome-invariant)
e_hat = cross_val_predict(clone(MODEL_T), W, t, groups=groups, cv=cf, n_jobs=-1,
                          method="predict_proba")[:, 1]
T_res = t - e_hat
print(f"propensity once: e in [{e_hat.min():.3f}, {e_hat.max():.3f}]")

propensity once: e in [0.001, 0.188]


## ATE (partial-linear, orthogonal score) per DV

In [4]:
def ate(Y_res):
    th=(Y_res*T_res).sum()/(T_res**2).sum(); J=(T_res**2).mean()
    IC=((Y_res-th*T_res)*T_res)/J; n=len(IC)
    se=np.sqrt((pd.Series(IC).groupby(groups).sum().values**2).sum())/n
    z=th/se; return th,se,z,2*(1-norm.cdf(abs(z)))

Yres_cache={}
rows=[]
for dv in DVS:
    y=df[dv].astype(float).values
    Yres_cache[dv]=y-cross_val_predict(clone(MODEL_Y),W,y,groups=groups,cv=cf,n_jobs=-1)
    th,se,z,p=ate(Yres_cache[dv]); base=y[t==0].mean()
    rows.append({"DV":dv,"control_mean":round(base,3),"ATE":round(th,4),"SE":round(se,4),
                 "z":round(z,2),"p":round(p,4),"rel_%":round(100*th/base,1)})
ate_tbl=pd.DataFrame(rows)
print(ate_tbl.to_string(index=False))

                   DV  control_mean     ATE     SE     z      p  rel_%
      post_n_pressure         2.145 -0.0788 0.0373 -2.12 0.0344   -3.7
        post_n_tackle         0.257 -0.0005 0.0112 -0.04 0.9674   -0.2
post_n_foul_committed         0.164 -0.0259 0.0085 -3.04 0.0024  -15.8
    post_n_def_events         2.567 -0.1081 0.0439 -2.46 0.0138   -4.2


**Reading.** ATEs are the headline causal estimates per DV (partial-linear,
orthogonal score, match-clustered SE). Heterogeneity is examined below via
pre-specified subgroup splits.

## Subgroup ATEs — moderator-driven heterogeneity analysis

The heterogeneity analysis treats every column of `Z` as a moderator.
Whenever `Z` changes (see `analysis_config.py`), the moderator set, subgroup
tables, and the bar-chart figures rebuild automatically.

For each moderator we estimate, reusing the DML orthogonal residuals
(`Y_res` = Y − m̂(W), `T_res` = T − ê(W)):

- a subgroup ATE table within auto-derived levels:
  - binary dummy (0/1) → its two original categorical levels
  - `ht_score_diff` → trailing / level / leading (natural cut at zero)
  - other continuous → median split (zero-cut if median is 0)
- a continuous linear interaction (`Y_res ~ T_res + T_res:Z`) for every
  non-binary column.

Each (DV × moderator) interaction p-value is corrected with Benjamini–Hochberg
across the family of tests.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# The paper reports heterogeneity for one outcome — fouls. Subgroup ATEs
# are estimated only on this DV across the five pre-specified moderators
# in Z (the half-time score differential plus four categoricals).
DV_HET = "post_n_foul_committed"

CAT_LEVELS = {c: sorted(df[c].astype(str).unique().tolist()) for c in CATS}

def derive_moderators_from_Z(Z_cols):
    """Reconstruct multi-level categoricals from their get_dummies columns;
    every continuous Z column is its own moderator."""
    out, seen = [], set()
    for col in Z_cols:
        cat = next((c for c in sorted(CATS, key=len, reverse=True)
                    if col.startswith(c + "_")), None)
        if cat:
            if cat not in seen:
                seen.add(cat); out.append({"name": cat, "kind": "categorical"})
        else:
            out.append({"name": col, "kind": "continuous"})
    return out

def group_for(mod):
    if mod["kind"] == "categorical":
        return df[mod["name"]].astype(str).values, CAT_LEVELS[mod["name"]]
    col = mod["name"]; x = Z[col]
    if col == "ht_score_diff":
        v = x.values
        return (np.where(v < 0, "trailing", np.where(v > 0, "leading", "level")),
                ["trailing", "level", "leading"])
    if x.nunique() <= 2:
        return x.astype(int).astype(str).values, [str(v) for v in sorted(x.unique())]
    m = x.median()
    return np.where(x.values > m, "high", "low"), ["low", "high"]

MIN_GROUP_N = 50

def subgroup_ate_for(dv, mod):
    g, levels = group_for(mod)
    counts = pd.Series(g).value_counts()
    if any(counts.get(lvl, 0) < MIN_GROUP_N for lvl in levels) or len(set(g)) < 2:
        return []
    name = mod["name"]
    d = pd.DataFrame({"Yr": Yres_cache[dv], "Tr": T_res, "m": groups,
                      "y": df[dv].astype(float).values, "t": t,
                      "g": pd.Categorical(g, categories=levels, ordered=False)})
    fit  = smf.ols("Yr ~ Tr:C(g) - 1", data=d).fit(cov_type="cluster", cov_kwds={"groups": d["m"]})
    fit2 = smf.ols("Yr ~ Tr + Tr:C(g)", data=d).fit(cov_type="cluster", cov_kwds={"groups": d["m"]})
    inter = [p for p in fit2.params.index if p.startswith("Tr:C(g)")]
    R = np.zeros((len(inter), len(fit2.params)))
    for i, nm in enumerate(inter):
        R[i, list(fit2.params.index).index(nm)] = 1
    p_int = float(fit2.f_test(R).pvalue)
    rows = []
    for lvl in levels:
        coef = f"Tr:C(g)[{lvl}]"
        if coef not in fit.params.index: continue
        mask = (g == lvl)
        cm = d.loc[mask & (d.t == 0), "y"].mean()
        rows.append({"moderator": name, "level": str(lvl),
                     "n": int(mask.sum()), "n_treated": int((mask & (d.t == 1)).sum()),
                     "ctrl_mean": round(cm, 3), "ATE": round(fit.params[coef], 4),
                     "SE": round(fit.bse[coef], 4), "p": round(fit.pvalues[coef], 4),
                     "interaction_p": round(p_int, 4)})
    return rows

MODERATORS = derive_moderators_from_Z(Z.columns)
cat_rows = []
for mod in MODERATORS:
    cat_rows += subgroup_ate_for(DV_HET, mod)

subgroup_tbl = pd.DataFrame(cat_rows)

inter_tbl = (subgroup_tbl[["moderator","interaction_p"]]
             .drop_duplicates().reset_index(drop=True))
inter_tbl["interaction_q_BH"] = multipletests(inter_tbl["interaction_p"], method="fdr_bh")[1].round(4)

pd.set_option("display.width", 200)
print(f"Subgroup ATEs on {DV_HET} across {len(MODERATORS)} moderators:")
print(subgroup_tbl.to_string(index=False))
print("\nJoint interaction tests (BH across the moderator family):")
print(inter_tbl.sort_values("interaction_q_BH").to_string(index=False))


**Reading.** The moderator set is the columns of `Z`; the analysis re-runs
whenever `Z` changes. Bar-chart figures (one per family) are produced by
`build_subgroup_figure.py` from the same definition.